# Практическая работа № 2. Работа с LLM через API

Вариант 19. Задание № 1. Извлечение общего числа проживающих и оценка точности.

Модель `Qwen3-4B` (Q4_K_M) вызывается через локальный HTTP API `llama.cpp`.

In [1]:
import httpx
import pandas as pd
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompt_values import StringPromptValue
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

MODEL = "Qwen3-4B-Q4_K_M"
API_URL = "http://127.0.0.1:18765/v1/chat/completions"


def make_model(output_schema=None):
    def invoke(prompt):
        roles = {"human": "user", "system": "system"}
        messages = [
            {"role": roles[message.type], "content": message.content}
            for message in prompt.to_messages()
        ]
        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": 0.0,
            "max_tokens": 384 if output_schema is not None else 128,
            "chat_template_kwargs": {"enable_thinking": False},
        }
        if output_schema is not None:
            payload["response_format"] = {"type": "json_object", "schema": output_schema}

        response = httpx.post(API_URL, json=payload, timeout=90.0, trust_env=False)
        response.raise_for_status()
        completion = response.json()["choices"][0]
        if completion.get("finish_reason") == "length":
            raise ValueError("Ответ превысил лимит токенов.")
        return AIMessage(content=completion["message"]["content"])

    return RunnableLambda(invoke)


def describe_error(error):
    if isinstance(error, httpx.HTTPStatusError):
        return f"HTTP {error.response.status_code}"
    return type(error).__name__

In [2]:
llm = make_model()
try:
    greeting = StringPromptValue(text="Ответь одним коротким приветствием на русском.")
    response = llm.invoke(greeting)
except Exception as error:
    raise RuntimeError(f"Проверка API {MODEL}: {describe_error(error)}. Запустите сервер по README.") from None
print(f"{MODEL} через llama.cpp: {response.content}")

Qwen3-4B-Q4_K_M через llama.cpp: Здравствуйте!


## Базовый промпт

В словаре `test_texts` ключи соответствуют номерам заявок, а значения содержат исходные
тексты. Цепочка передаёт модели только текст заявки. Колонка `amount` нужна при оценке.

In [3]:
df = pd.read_csv("rental_19.csv", sep=";", encoding="utf-8-sig")
if list(df.columns) != ["amount", "text"] or len(df) != 15 or df.isna().any().any():
    raise ValueError("Ожидаются 15 заполненных заявок со столбцами amount и text.")
df.index = pd.RangeIndex(1, len(df) + 1, name="id")
test_texts = df["text"].to_dict()
df.head()

,amount,text
id,,
1,2,Сниму жильё в Аше ( на улице Репина)\r\nС 16/2...
2,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...
3,4,Ищем жильё с15.06 на 2 недели 2 взрослыми 2 де...
4,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но..."
5,2,"Здравствуйте, интересует жильё с 7.09 по 14.09..."


In [4]:
basic_prompt = PromptTemplate.from_template(
    "Проанализируй заявку на аренду жилья и определи количество проживающих. "
    "Если число не указано прямо, определи его по контексту. "
    "Верни только одно целое число без пояснений.\n\n"
    "Заявка:\n{text}\n\nКоличество человек:"
)
chain = basic_prompt | llm | StrOutputParser()

In [5]:
results = []
for text in test_texts.values():
    try:
        result = chain.invoke({"text": text})
    except Exception as error:
        result = f"ERROR: {describe_error(error)}"
    results.append(result)

df["result"] = results
df[["amount", "result"]]

,amount,result
id,,
1,2,2
2,6,10
3,4,6
4,2,2
5,2,2
6,3,4
7,3,3
8,4,4
9,4,4


## Точность на 15 заявках

Ответ считается верным, если целое число совпало с `amount`. Ошибка API или ответ,
который нельзя прочитать как целое число, засчитывается как ошибка.

In [6]:
raw = df["result"].astype("string").str.strip()
df["result_numeric"] = pd.to_numeric(
    raw.where(raw.str.fullmatch(r"[0-9]+")), errors="coerce"
)
df["correct"] = df["result_numeric"].eq(df["amount"]).fillna(False)

total = len(df)
correct = int(df["correct"].sum())
print(f"Верных ответов: {correct} из {total}")
print(f"Ошибок: {total - correct}")
print(f"Точность: {correct / total:.1%}")

df.to_csv("rental_with_results.csv", encoding="utf-8-sig")
df.loc[~df["correct"], ["text", "amount", "result"]]

Верных ответов: 9 из 15
Ошибок: 6
Точность: 60.0%


,text,amount,result
id,,,
2,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,10
3,Ищем жильё с15.06 на 2 недели 2 взрослыми 2 де...,4,6
6,Добрый вечер. Ищем жилье с 17.08 на 10 дней. Ж...,3,4
10,Снимем жилье в Лазаревском с 30.07 по 09.08.15...,5,6
11,Ищем жильё с 4.07.на 20 дней. 2 взрослых и 2 д...,4,6
14,Здравствуйте!Ищем жильё в Лазаревское.2 взросл...,3,4
